# 🎓 第2章 第3课：异象检验 + GRS + 正交化

**本节目标**：当一次医生，做三件事。

1. **造一个异象**：给某资产注入真实 α，看时序回归能不能把它揪出来（含 NW 修正）；
2. **GRS 体检**：三个因子够不够？
3. **正交化**：去掉重复信息后，因子还剩多少增量。

对应原书 2.4~2.6 节。

In [1]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src' / 'factor_zero').exists():
            return p
    raise FileNotFoundError('找不到仓库根目录，请从仓库内启动 notebook')
ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factor_zero.sorting import quantile_group, sort_portfolios, spread_return, monotonicity_test
from factor_zero.factors import monthly_returns, momentum, book_to_market, log_market_cap
from factor_zero.statistics import nw_t_stat

def _find_sample():
    for p in [ROOT / 'data' / 'sample', Path('data/sample'), Path.cwd().parents[2] / 'data' / 'sample']:
        if p.exists():
            return p
    raise FileNotFoundError('找不到 data/sample，请从仓库内启动 notebook')
SAMPLE = _find_sample()
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
print('✅ 环境就绪，开始学习！')

from factor_zero.statistics import newey_west_vcov, grs_test
from factor_zero.regression import time_series_betas

✅ 环境就绪，开始学习！


## 0) 造一个异象

8 个资产、3 个因子，给资产 0 注入 **0.8%/月** 的真实 α，其余为 0。
如果方法有效，时序回归应该能在资产 0 上看到显著 α；但即使真实 α=0，也可能有约 5% 的假阳性——所以还要看 GRS 和多重检验。

In [2]:
rng = np.random.default_rng(99)
T, N, K = 240, 8, 3
beta = rng.uniform(0.5, 1.5, size=(N, K))
lam = np.array([0.006, 0.004, 0.002])
f = rng.multivariate_normal(lam, np.diag([0.02**2] * K), size=T)
alpha_true = np.zeros(N)
alpha_true[0] = 0.008
r = alpha_true + f @ beta.T + rng.normal(0, 0.01, size=(T, N))
print('注入真相：资产0 α=0.8%/月，其余为0')

注入真相：资产0 α=0.8%/月，其余为0


## 1) 时序回归检验异象（2.4.1 节）

对每个资产回归 $r = lpha + eta f + arepsilon$。
对比 **OLS 标准误 vs Newey-West 标准误**。

In [3]:
fdf = pd.DataFrame(f, columns=['F1', 'F2', 'F3'])
alphas, betas, resid = time_series_betas(pd.DataFrame(r, columns=[f'a{i}' for i in range(N)]), fdf)
rows = []
X = np.column_stack([np.ones(T), f])
for i in range(N):
    e = resid.iloc[:, i].to_numpy()
    vcov_ols = np.linalg.inv(X.T @ X) * (e @ e / (T - K - 1))
    vcov_nw = newey_west_vcov(X, e)
    rows.append({
        '资产': f'a{i}',
        'α估计(%/月)': alphas.iloc[i] * 100,
        'OLS t': alphas.iloc[i] / np.sqrt(vcov_ols[0, 0]),
        'NW t': alphas.iloc[i] / np.sqrt(vcov_nw[0, 0]),
        '真相α(%/月)': alpha_true[i] * 100,
    })
pd.DataFrame(rows).round(3)

,资产,α估计(%/月),OLS t,NW t,真相α(%/月)
0,a0,0.737,11.548,12.115,0.8
1,a1,0.039,0.565,0.622,0.0
2,a2,-0.037,-0.556,-0.559,0.0
3,a3,0.067,1.066,1.084,0.0
4,a4,-0.016,-0.238,-0.258,0.0
5,a5,-0.165,-2.295,-2.293,0.0
6,a6,0.156,2.354,2.387,0.0
7,a7,0.028,0.374,0.327,0.0


> 🔍 **注意假阳性**：即使真实 α=0，8 个资产里也可能有 1–2 个 |t| 略大于 2 的“假阳性”（约 5% 概率）。
> 所以不能逐个看 t 值就下结论，还要看 **GRS 联合检验**和多重检验校正。


## 2) GRS 体检：3 因子模型够吗？（2.5.1 节）

GRS 联合检验所有 α 是否为零。
异象存在 → 模型被拒绝（p 值小）。

In [4]:
F, p = grs_test(alphas.to_numpy(), resid.to_numpy(), f)
print(f'GRS F = {F:.3f}, p 值 = {p:.6f}')
print('p < 0.05 → 模型被拒绝：存在模型解释不了的异象' if p < 0.05 else 'p ≥ 0.05 → 模型通过体检')

GRS F = 17.934, p 值 = 0.000000
p < 0.05 → 模型被拒绝：存在模型解释不了的异象


## 3) 正交化：只听增量信息（2.6 节）

假设 F2 和 F3 高度相关——把 F3 对 F2 回归，**残差 = F3 的纯新增信息**。
正交化后，F3 与 F2 的相关系数应接近 0。

In [5]:
f2 = f[:, 1]
f3_corr = 0.9 * f2 + rng.normal(0, 0.01, T)  # 人为制造高度相关
coef = np.polyfit(f2, f3_corr, 1)
f3_orth = f3_corr - (coef[0] * f2 + coef[1])
print(f'正交化前 F2 与 F3 的相关系数: {np.corrcoef(f2, f3_corr)[0, 1]:.4f}')
print(f'正交化后 F2 与 F3 的相关系数: {np.corrcoef(f2, f3_orth)[0, 1]:.10f}')
print('→ 正交化后相关性归零，残差就是 F3 独有的信息')

正交化前 F2 与 F3 的相关系数: 0.8998
正交化后 F2 与 F3 的相关系数: 0.0000000000
→ 正交化后相关性归零，残差就是 F3 独有的信息


## ✋ 试一试

1. 把 `alpha_true[0]` 改成 0，GRS 的 p 值会怎么变？
2. 把注入 α 改成 0.003（0.3%/月），NW t 还能显著吗？
3. 把 `f3_corr = 0.9*f2 + ...` 的 0.9 改成 0.3，正交化前后相关系数怎么变？

## ✅ 本节小结

- **异象检验** = 回归 α 显著 ≠ 0（别忘了 NW 眼镜）；
- **GRS** = 把一群资产的 α 打包查，模型好不好看 p 值；
- **正交化** = 残差即增量，业界构建因子前的标准动作。

> 下一章：把这些方法用到 A 股七大因子上（第3章）。